# Sentiment Analysis on Twitter Dataset

EDA + Machine Learning + LLM

## Dataset Description
- Train: 27,481 samples
- Test: 3,534 samples
- Labels: positive, negative, neutral


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print(train_df.shape)
print(test_df.shape)
train_df.head()

# Part A - EDA

In [ ]:
train_df.info()
train_df.describe(include="all")

In [ ]:
train_df.isnull().sum()

In [ ]:
train_df["sentiment"].value_counts()

In [ ]:
sns.countplot(data=train_df,x="sentiment")
plt.title("Sentiment Distribution")
plt.show()

In [ ]:
train_df["length"] = train_df["text"].astype(str).apply(len)

sns.histplot(train_df["length"], bins=50)
plt.title("Tweet Length Distribution")
plt.show()

In [ ]:
from wordcloud import WordCloud
positive_text=" ".join(train_df[train_df.sentiment=="positive"]["text"].astype(str))
wc=WordCloud(width=800,height=400,background_color="white").generate(positive_text)
plt.imshow(wc)
plt.axis("off")
plt.show()

# Part B - Machine Learning

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer=TfidfVectorizer(max_features=10000,stop_words="english")
X_train=vectorizer.fit_transform(train_df["text"].astype(str))
X_test=vectorizer.transform(test_df["text"].astype(str))

y_train=train_df["sentiment"]
y_test=test_df["sentiment"]

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr=LogisticRegression(max_iter=1000)
lr.fit(X_train,y_train)
lr_pred=lr.predict(X_test)

print(classification_report(y_test,lr_pred))

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb=MultinomialNB()
nb.fit(X_train,y_train)
nb_pred=nb.predict(X_test)

print(classification_report(y_test,nb_pred))

In [ ]:
from sklearn.svm import LinearSVC

svm=LinearSVC()
svm.fit(X_train,y_train)
svm_pred=svm.predict(X_test)

print(classification_report(y_test,svm_pred))

# Part C - Large Language Model

In [ ]:
from transformers import pipeline

pipe=pipeline("sentiment-analysis",model="cardiffnlp/twitter-roberta-base-sentiment-latest")

In [ ]:
pipe("I love this movie")

In [ ]:
from tqdm import tqdm

label_map={"negative":"negative","neutral":"neutral","positive":"positive"}
llm_pred=[]

for text in tqdm(test_df["text"].astype(str)):
    result=pipe(text)[0]
    llm_pred.append(label_map[result["label"].lower()])

print(classification_report(y_test,llm_pred))

# Part D - Model Comparison

In [ ]:
from sklearn.metrics import accuracy_score

results=pd.DataFrame({
"Model":["Logistic Regression","Naive Bayes","SVM"],
"Accuracy":[accuracy_score(y_test,lr_pred),accuracy_score(y_test,nb_pred),accuracy_score(y_test,svm_pred)]
})
results

In [ ]:
sns.barplot(data=results,x="Model",y="Accuracy")
plt.title("Model Comparison")
plt.show()

## Conclusion
EDA, Logistic Regression, Naive Bayes, SVM và RoBERTa sentiment model đã được áp dụng để so sánh hiệu quả phân loại cảm xúc.